# CLT-Forge Training on Custom NanoGPT

End-to-end notebook for training a Cross-Layer Transcoder (CLT) on your locally saved NanoGPT model.

**Architecture is read directly from your checkpoint — no hardcoding needed.**

**Pipeline stages:**
1. Install dependencies
2. Configuration — set paths and CLT hyperparameters only
3. Convert NanoGPT dataset to HuggingFace Arrow format (run once)
4. Load checkpoint and read architecture from `model_args`
5. Convert NanoGPT weights to TransformerLens HookedTransformer
6. Verify the converted model
7. Move to device and freeze
8. Build CLT-Forge config
9. Precompute and cache MLP activations
10. Train the CLT
11. Evaluate reconstruction quality
12. Run AutoInterp (optional)
13. Compute Attribution Graph (optional)
14. Launch Visual Interface (optional)

---
## Cell 1 — Install Dependencies

In [ ]:
import subprocess, sys

# Install CLT-Forge from your local clone (editable install)
CLT_FORGE_REPO_PATH = "/path/to/your/CLT-Forge"   # <-- CHANGE THIS to your local clone
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", CLT_FORGE_REPO_PATH, "-q"])

# Core dependencies
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformer_lens",
    "tiktoken",
    "datasets",
    "wandb",
    "zstandard",
])

print('All dependencies installed.')

---
## Cell 2 — Configuration

**Only edit this cell.**  
Architecture values (`N_LAYER`, `N_HEAD`, `N_EMBD`, `BLOCK_SIZE`, `VOCAB_SIZE`) are read
automatically from your checkpoint in Cell 4 — do not set them here.

**To use your own training dataset** (recommended), set `USE_LOCAL_DATASET = True`
and point `LOCAL_DATASET_BIN` at your NanoGPT `.bin` file. Cell 3 will convert it
to HuggingFace Arrow format once, then CLT-Forge will use it directly.

In [ ]:
import os, torch

# ── Paths ──────────────────────────────────────────────────────────────────────
NANOGPT_CKPT_PATH      = "out/ckpt.pt"             # <-- your checkpoint
CONVERTED_MODEL_PATH   = "./converted_tl_model.pt"
CACHED_ACTIVATIONS_DIR = "./cached_activations"
CLT_CHECKPOINT_DIR     = "./clt_checkpoints"
AUTOINTERP_DIR         = "./autointerp_results"
ATTRIBUTION_GRAPH_DIR  = "./attribution_graphs"

for d in [CACHED_ACTIVATIONS_DIR, CLT_CHECKPOINT_DIR, AUTOINTERP_DIR, ATTRIBUTION_GRAPH_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Dataset ────────────────────────────────────────────────────────────────────
# Option A: use your own NanoGPT training data (recommended)
#   Set USE_LOCAL_DATASET = True and point LOCAL_DATASET_BIN at your .bin file.
#   Cell 3 will convert it to HuggingFace Arrow format and save to LOCAL_DATASET_ARROW.
#
# Option B: use a HuggingFace hub dataset
#   Set USE_LOCAL_DATASET = False and set HUB_DATASET_PATH below.

USE_LOCAL_DATASET   = True
LOCAL_DATASET_BIN   = "data/your_dataset/train.bin"    # <-- your NanoGPT .bin file
LOCAL_DATASET_ARROW = "./hf_arrow_dataset"             # where the converted dataset is saved
HUB_DATASET_PATH    = "apollo-research/Skylion007-openwebtext-tokenizer-gpt2"  # used if USE_LOCAL_DATASET=False

# ── CLT training hyperparameters ───────────────────────────────────────────────
EXPANSION_FACTOR        = 16     # CLT features = N_EMBD * EXPANSION_FACTOR
CONTEXT_SIZE            = 64     # tokens per sample — must be <= BLOCK_SIZE
TOTAL_TRAINING_TOKENS   = 5_000_000
TRAIN_BATCH_SIZE_TOKENS = 2048
GRAD_ACCUM_STEPS        = 4
LEARNING_RATE           = 4e-4
L0_COEFFICIENT          = 2.0
USE_COMPRESSION         = True

# ── Device ─────────────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device             : {DEVICE}")
print(f"Use local dataset  : {USE_LOCAL_DATASET}")
print("Architecture will be read from checkpoint in Cell 4.")

---
## Cell 3 — Convert NanoGPT Dataset to HuggingFace Arrow Format

CLT-Forge's `ActivationsStore` requires a HuggingFace dataset with a `tokens` column.
NanoGPT stores data as raw `.bin` (numpy memmap) files — this cell converts it.

**Run once.** Skipped automatically if the Arrow dataset already exists,
or if `USE_LOCAL_DATASET = False`.

In [ ]:
import os, numpy as np
from datasets import Dataset

if not USE_LOCAL_DATASET:
    DATASET_PATH = HUB_DATASET_PATH
    DISK         = False
    print(f"Using HuggingFace hub dataset: {HUB_DATASET_PATH}")

elif os.path.exists(LOCAL_DATASET_ARROW):
    DATASET_PATH = LOCAL_DATASET_ARROW
    DISK         = True
    print(f"Arrow dataset already exists at: {LOCAL_DATASET_ARROW} — skipping conversion.")

else:
    print(f"Converting {LOCAL_DATASET_BIN} to HuggingFace Arrow format...")
    print(f"  context_size = {CONTEXT_SIZE} (uses BLOCK_SIZE after Cell 4; using CONTEXT_SIZE now)")

    # NanoGPT saves tokens as uint16 memmap
    data = np.memmap(LOCAL_DATASET_BIN, dtype=np.uint16, mode="r")

    # Chunk into sequences of CONTEXT_SIZE
    n_sequences = len(data) // CONTEXT_SIZE
    if n_sequences == 0:
        raise ValueError(
            f"{LOCAL_DATASET_BIN} has only {len(data)} tokens — not enough for "
            f"even one sequence of length {CONTEXT_SIZE}."
        )

    tokens_list = [
        data[i * CONTEXT_SIZE : (i + 1) * CONTEXT_SIZE].tolist()
        for i in range(n_sequences)
    ]

    ds = Dataset.from_dict({"tokens": tokens_list})
    ds.save_to_disk(LOCAL_DATASET_ARROW)

    DATASET_PATH = LOCAL_DATASET_ARROW
    DISK         = True
    print(f"Saved {len(ds):,} sequences ({n_sequences * CONTEXT_SIZE:,} tokens) to {LOCAL_DATASET_ARROW}")

print(f"\nDATASET_PATH = {DATASET_PATH}")
print(f"DISK         = {DISK}")

---
## Cell 4 — Load Checkpoint and Read Architecture

Reads `model_args` directly from your checkpoint so `N_LAYER`, `N_HEAD`, `N_EMBD`,
`BLOCK_SIZE`, and `VOCAB_SIZE` always match what you actually trained.

Also strips the `_orig_mod.` prefix added by `torch.compile` and detects
whether you used a custom vocab (`meta.pkl`) or the standard GPT-2 BPE tokeniser.

In [ ]:
import os, pickle, torch

print(f"Loading checkpoint: {NANOGPT_CKPT_PATH}")
checkpoint = torch.load(NANOGPT_CKPT_PATH, map_location="cpu", weights_only=False)

print("Checkpoint keys:", list(checkpoint.keys()))
for key in ("iter_num", "best_val_loss"):
    if key in checkpoint:
        print(f"  {key}: {checkpoint[key]}")

# ── Read architecture from saved model_args ────────────────────────────────────
model_args = checkpoint["model_args"]
print(f"\nmodel_args: {model_args}")

N_LAYER    = model_args["n_layer"]
N_HEAD     = model_args["n_head"]
N_EMBD     = model_args["n_embd"]
BLOCK_SIZE = model_args["block_size"]
BIAS       = model_args.get("bias", True)

# ── Extract state dict — strip torch.compile and DDP prefixes ─────────────────
raw_sd = checkpoint["model"]
raw_sd = {
    (k[len("_orig_mod."):] if k.startswith("_orig_mod.") else k): v
    for k, v in raw_sd.items()
}
raw_sd = {
    (k[len("module."):] if k.startswith("module.") else k): v
    for k, v in raw_sd.items()
}

# ── Detect tokeniser: custom meta.pkl vs GPT-2 BPE ────────────────────────────
dataset   = checkpoint.get("config", {}).get("dataset", None)
meta_path = os.path.join("data", dataset, "meta.pkl") if dataset else None

if meta_path and os.path.exists(meta_path):
    print(f"\nCustom vocab found: {meta_path}")
    with open(meta_path, "rb") as f:
        meta = pickle.load(f)
    VOCAB_SIZE        = meta["vocab_size"]
    TL_TOKENIZER_NAME = None
    print(f"Custom vocab size : {VOCAB_SIZE}")
else:
    print("\nNo meta.pkl found — using GPT-2 BPE tokeniser")
    VOCAB_SIZE        = model_args.get("vocab_size", 50304)
    TL_TOKENIZER_NAME = "gpt2"

print(f"\nArchitecture confirmed:")
print(f"  n_layer    = {N_LAYER}")
print(f"  n_head     = {N_HEAD}")
print(f"  n_embd     = {N_EMBD}")
print(f"  block_size = {BLOCK_SIZE}")
print(f"  vocab_size = {VOCAB_SIZE}")
print(f"  bias       = {BIAS}")
print(f"  tokeniser  = {TL_TOKENIZER_NAME or 'custom (meta.pkl)'}")
print(f"\nParameter tensors: {len(raw_sd)}")
print("First 12 keys:")
for k in list(raw_sd.keys())[:12]:
    print(f"  {k:60s} {tuple(raw_sd[k].shape)}")

---
## Cell 5 — Convert NanoGPT Weights to TransformerLens Format

Key shape transformations:
- NanoGPT `c_attn` packs Q, K, V into one `(3*d_model, d_model)` matrix — split and reshape each to `(n_heads, d_model, d_head)`
- NanoGPT uses Conv1D convention `(out, in)` — TransformerLens uses `(in, out)` for W_in / W_out
- All bias keys use `.get()` with zero fallbacks to handle `bias=False` models

In [ ]:
import torch
from transformer_lens import HookedTransformer, HookedTransformerConfig

# Build TransformerLens config from checkpoint-derived values
tl_cfg = HookedTransformerConfig(
    n_layers           = N_LAYER,
    d_model            = N_EMBD,
    n_heads            = N_HEAD,
    d_head             = N_EMBD // N_HEAD,
    d_mlp              = 4 * N_EMBD,
    n_ctx              = BLOCK_SIZE,
    d_vocab            = VOCAB_SIZE,
    act_fn             = "gelu_new",
    normalization_type = "LN",
    tokenizer_name     = TL_TOKENIZER_NAME,
    device             = "cpu",
    attn_only          = False,
)

def convert_nanogpt_weights(sd: dict, cfg: HookedTransformerConfig) -> dict:
    """
    Remap a NanoGPT state_dict to the TransformerLens HookedTransformer format.
    All bias keys use .get() with zero fallbacks to handle bias=False models.
    """
    tl = {}
    d  = cfg.d_model
    nh = cfg.n_heads
    dh = cfg.d_head

    # Token + position embeddings (weights always exist)
    tl["embed.W_E"]       = sd["transformer.wte.weight"]
    tl["pos_embed.W_pos"] = sd["transformer.wpe.weight"]

    # Final LayerNorm
    tl["ln_final.w"] = sd["transformer.ln_f.weight"]
    tl["ln_final.b"] = sd.get("transformer.ln_f.bias", torch.zeros(cfg.d_model))

    # Unembed — NanoGPT ties lm_head.weight == wte.weight by default
    lm_w = sd.get("lm_head.weight", sd["transformer.wte.weight"])
    tl["unembed.W_U"] = lm_w.T
    tl["unembed.b_U"] = torch.zeros(cfg.d_vocab)

    for L in range(cfg.n_layers):
        p = f"transformer.h.{L}"

        # LayerNorm 1 (before attention)
        tl[f"blocks.{L}.ln1.w"] = sd[f"{p}.ln_1.weight"]
        tl[f"blocks.{L}.ln1.b"] = sd.get(f"{p}.ln_1.bias", torch.zeros(cfg.d_model))

        # ── Attention QKV ──────────────────────────────────────────────────
        # NanoGPT Conv1D: c_attn.weight (3d, d) — split into Q, K, V
        c_attn_w = sd[f"{p}.attn.c_attn.weight"]
        c_attn_b = sd.get(f"{p}.attn.c_attn.bias", torch.zeros(3 * d))

        W_Q_flat, W_K_flat, W_V_flat = c_attn_w.split(d, dim=0)
        b_Q_flat, b_K_flat, b_V_flat = c_attn_b.split(d, dim=0)

        # Reshape to (n_heads, d_model, d_head)
        tl[f"blocks.{L}.attn.W_Q"] = W_Q_flat.T.reshape(d, nh, dh).permute(1, 0, 2)
        tl[f"blocks.{L}.attn.W_K"] = W_K_flat.T.reshape(d, nh, dh).permute(1, 0, 2)
        tl[f"blocks.{L}.attn.W_V"] = W_V_flat.T.reshape(d, nh, dh).permute(1, 0, 2)

        tl[f"blocks.{L}.attn.b_Q"] = b_Q_flat.reshape(nh, dh)
        tl[f"blocks.{L}.attn.b_K"] = b_K_flat.reshape(nh, dh)
        tl[f"blocks.{L}.attn.b_V"] = b_V_flat.reshape(nh, dh)

        # ── Attention output projection ─────────────────────────────────────
        # c_proj.weight: (d, d) Conv1D layout — TL W_O: (n_heads, d_head, d_model)
        c_proj_w = sd[f"{p}.attn.c_proj.weight"]
        c_proj_b = sd.get(f"{p}.attn.c_proj.bias", torch.zeros(cfg.d_model))
        tl[f"blocks.{L}.attn.W_O"] = c_proj_w.T.reshape(nh, dh, d)
        tl[f"blocks.{L}.attn.b_O"] = c_proj_b

        # LayerNorm 2 (before MLP)
        tl[f"blocks.{L}.ln2.w"] = sd[f"{p}.ln_2.weight"]
        tl[f"blocks.{L}.ln2.b"] = sd.get(f"{p}.ln_2.bias", torch.zeros(cfg.d_model))

        # ── MLP ─────────────────────────────────────────────────────────────
        # NanoGPT Conv1D: c_fc (4d, d), c_proj (d, 4d)
        # TL: W_in (d, 4d), W_out (4d, d)
        tl[f"blocks.{L}.mlp.W_in"]  = sd[f"{p}.mlp.c_fc.weight"].T
        tl[f"blocks.{L}.mlp.b_in"]  = sd.get(f"{p}.mlp.c_fc.bias",   torch.zeros(cfg.d_mlp))
        tl[f"blocks.{L}.mlp.W_out"] = sd[f"{p}.mlp.c_proj.weight"].T
        tl[f"blocks.{L}.mlp.b_out"] = sd.get(f"{p}.mlp.c_proj.bias", torch.zeros(cfg.d_model))

    return tl


print("Converting weights...")
tl_state_dict = convert_nanogpt_weights(raw_sd, tl_cfg)
print(f"Converted {len(tl_state_dict)} parameter tensors.")
print("Sample keys:", list(tl_state_dict.keys())[:8])

---
## Cell 6 — Build the HookedTransformer and Verify

Loads converted weights into a blank HookedTransformer and runs a forward pass.
Missing keys in the output are causal mask buffers — expected and harmless.

In [ ]:
from transformer_lens import HookedTransformer

tl_model = HookedTransformer(tl_cfg)
tl_model.eval()

# strict=False tolerates missing causal mask buffers (blocks.L.attn.mask / IGNORE)
missing, unexpected = tl_model.load_state_dict(tl_state_dict, strict=False)

if missing:
    print("Missing keys (causal mask buffers — expected and OK):")
    for k in missing[:10]: print(f"  {k}")
if unexpected:
    print("Unexpected keys:")
    for k in unexpected[:10]: print(f"  {k}")

# Forward pass sanity check
test_tokens = torch.randint(0, VOCAB_SIZE, (1, 16))
with torch.no_grad():
    logits = tl_model(test_tokens)
assert logits.shape == (1, 16, VOCAB_SIZE), f"Unexpected logit shape: {logits.shape}"
print(f"Forward pass OK — logits shape: {logits.shape}")

# Verify MLP hook points (ActivationsStore needs these)
print("\nMLP hook points:")
for L in range(N_LAYER):
    pre_key  = f"blocks.{L}.mlp.hook_pre"
    post_key = f"blocks.{L}.mlp.hook_post"
    has_pre  = pre_key  in tl_model.hook_dict
    has_post = post_key in tl_model.hook_dict
    print(f"  Layer {L}: hook_pre={has_pre}, hook_post={has_post}")
    assert has_pre and has_post, f"Missing MLP hook at layer {L}"

print("\nAll MLP hook points verified.")
torch.save(tl_state_dict, CONVERTED_MODEL_PATH)
print(f"Converted weights saved: {CONVERTED_MODEL_PATH}")

---
## Cell 7 — Move to Device and Freeze

In [ ]:
tl_model = tl_model.to(DEVICE)
tl_model.eval()

# Freeze — we only extract activations, never train the base model
for param in tl_model.parameters():
    param.requires_grad_(False)

param_count = sum(p.numel() for p in tl_model.parameters())
print(f"Model on device  : {next(tl_model.parameters()).device}")
print(f"Total parameters : {param_count:,}")

# Quick generation check — only works with GPT-2 BPE tokeniser
if TL_TOKENIZER_NAME is not None:
    try:
        with torch.no_grad():
            sample = tl_model.generate("The", max_new_tokens=8, temperature=1.0)
        print(f"Sample generation: {sample}")
    except Exception as e:
        print(f"Generation check skipped: {e}")
else:
    print("Generation check skipped (custom vocab — no string tokeniser).")

---
## Cell 8 — Build the CLT-Forge Training Config

Uses the architecture values read from the checkpoint in Cell 4.
Validated against the actual `CLTTrainingRunnerConfig` pydantic schema.

In [ ]:
from clt_forge.config import CLTTrainingRunnerConfig

gradient_accumulation_steps = GRAD_ACCUM_STEPS
effective_batch              = TRAIN_BATCH_SIZE_TOKENS * gradient_accumulation_steps
total_training_steps         = TOTAL_TRAINING_TOKENS // effective_batch

print(f"Effective batch size : {effective_batch} tokens")
print(f"Total training steps : {total_training_steps}")

cfg = CLTTrainingRunnerConfig(
    # System
    distributed_setup           = "None",          # capital-N string — single GPU; 'feature_sharding' for multi-GPU
    device                      = DEVICE,
    dtype                       = "float32",
    seed                        = 42,

    # Model and data
    model_name                  = TL_TOKENIZER_NAME or "gpt2",
    d_in                        = N_EMBD,
    expansion_factor            = EXPANSION_FACTOR,
    dataset_path                = DATASET_PATH,
    disk                        = DISK,            # True = load_from_disk, False = HuggingFace hub
    context_size                = CONTEXT_SIZE,
    cached_activations_path     = CACHED_ACTIVATIONS_DIR,
    n_train_batch_per_buffer    = 64,

    # Checkpointing
    checkpoint_path             = CLT_CHECKPOINT_DIR,
    from_pretrained_path        = None,

    # JumpReLU
    jumprelu_init_threshold     = 0.03,
    jumprelu_bandwidth          = 1.0,

    # Batching
    train_batch_size_tokens     = TRAIN_BATCH_SIZE_TOKENS,
    gradient_accumulation_steps = gradient_accumulation_steps,

    # Training duration
    total_training_tokens       = TOTAL_TRAINING_TOKENS,

    # Optimisation
    lr                          = LEARNING_RATE,
    lr_warm_up_steps            = min(1_000, total_training_steps // 10),
    lr_decay_steps              = total_training_steps // 20,
    adam_beta1                  = 0.0,
    adam_beta2                  = 0.999,

    # Sparsity
    l0_coefficient              = L0_COEFFICIENT,
    l0_warm_up_steps            = int(0.7 * total_training_steps),
    dead_penalty_coef           = 1e-5,
    dead_feature_window         = 250,

    # Logging — wandb off
    log_to_wandb                = False,
    wandb_project               = "nanogpt-clt",
    wandb_log_frequency         = 10,
    eval_every_n_wandb_logs     = 50,
    run_name                    = f"nanogpt-{N_LAYER}L-{N_EMBD}d-clt",
)

print("CLTTrainingRunnerConfig created successfully.")
print(f"  d_in      : {cfg.d_in}")
print(f"  d_latent  : {cfg.d_latent}")
print(f"  disk      : {cfg.disk}")
print(f"  dataset   : {cfg.dataset_path}")

---
## Cell 9 — Precompute and Cache MLP Activations

Runs the frozen NanoGPT over the dataset and saves MLP input/output pairs to disk.
**Run once.** Skips automatically if cached files already exist.

In [ ]:
import os
from clt_forge import ActivationsStore

existing = [f for f in os.listdir(CACHED_ACTIVATIONS_DIR)
            if f.endswith(".pt") or f.endswith(".zst")]

if existing:
    print(f"Found {len(existing)} cached activation files — skipping generation.")
    print("Delete the contents of", CACHED_ACTIVATIONS_DIR, "to regenerate.")
else:
    print(f"Generating activations for {TOTAL_TRAINING_TOKENS:,} tokens...")
    print(f"  Context size : {CONTEXT_SIZE}")
    print(f"  Compression  : {USE_COMPRESSION}")
    print(f"  Save path    : {CACHED_ACTIVATIONS_DIR}")

    store = ActivationsStore(model=tl_model, cfg=cfg)

    store.generate_and_save_activations(
        path            = cfg.cached_activations_path,
        use_compression = USE_COMPRESSION,
    )

    saved = [f for f in os.listdir(CACHED_ACTIVATIONS_DIR)
             if f.endswith(".pt") or f.endswith(".zst")]
    print(f"\nActivation caching complete. {len(saved)} files saved.")

---
## Cell 10 — Train the CLT

Trains the Cross-Layer Transcoder using the precomputed activation chunks.

The training objective minimises:
- **Reconstruction loss** — CLT outputs approximate MLP outputs across all layer pairs
- **Sparsity penalty (L1)** — encourages few active features per token via JumpReLU
- **Dead feature penalty** — prevents features from never activating

In [ ]:
import glob
from clt_forge import CLTTrainingRunner

print("Starting CLT training...")
print(f"  Layers    : {N_LAYER}")
print(f"  Features  : {N_EMBD * EXPANSION_FACTOR}")
print(f"  Tokens    : {cfg.total_training_tokens:,}")
print(f"  Ckpt dir  : {cfg.checkpoint_path}")
print()

trainer = CLTTrainingRunner(cfg)
trainer.run()

print("\nCLT training complete!")

checkpoints = sorted(glob.glob(os.path.join(CLT_CHECKPOINT_DIR, "**/*.pt"), recursive=True))
if checkpoints:
    BEST_CHECKPOINT = checkpoints[-1]
    print(f"Latest checkpoint : {BEST_CHECKPOINT}")
else:
    print("WARNING: No checkpoint files found — check checkpoint_path in config.")
    BEST_CHECKPOINT = CLT_CHECKPOINT_DIR

---
## Cell 11 — Evaluate Reconstruction Quality

Computes per-layer explained variance: how much of the MLP output variance the CLT captures.

In [ ]:
import torch

try:
    from clt_forge import CLT

    clt_model = CLT.from_pretrained(BEST_CHECKPOINT)
    clt_model.eval().to(DEVICE)
    print(f"Loaded CLT from: {BEST_CHECKPOINT}")

    eval_tokens = torch.randint(0, VOCAB_SIZE, (4, CONTEXT_SIZE), device=DEVICE)

    with torch.no_grad():
        _, cache = tl_model.run_with_cache(
            eval_tokens,
            names_filter = lambda n: "mlp" in n and ("hook_pre" in n or "hook_post" in n)
        )

    mlp_inputs  = [cache[f"blocks.{L}.mlp.hook_pre"]  for L in range(N_LAYER)]
    mlp_outputs = [cache[f"blocks.{L}.mlp.hook_post"] for L in range(N_LAYER)]

    with torch.no_grad():
        clt_recons = clt_model.forward(mlp_inputs)

    print("\nPer-layer explained variance (1.0 = perfect):")
    total_ev = 0.0
    for L in range(N_LAYER):
        target = mlp_outputs[L].float()
        pred   = clt_recons[L].float()
        ss_res = ((target - pred) ** 2).sum().item()
        ss_tot = ((target - target.mean()) ** 2).sum().item()
        ev = 1.0 - ss_res / (ss_tot + 1e-8)
        total_ev += ev
        bar = '#' * int(max(0, ev) * 30)
        print(f"  Layer {L}: {ev:.4f}  |{bar:<30}|")
    print(f"\n  Mean EV: {total_ev / N_LAYER:.4f}")

except Exception as e:
    print(f"Evaluation skipped — re-run after training completes.")
    print(f"Error: {e}")

---
## Cell 12 — (Optional) Run AutoInterp

Set `RUN_AUTOINTERP = True` to enable.

In [ ]:
RUN_AUTOINTERP = False

if RUN_AUTOINTERP:
    from clt_forge import AutoInterp, AutoInterpConfig

    autointerp_cfg = AutoInterpConfig(
        device                  = DEVICE,
        model_name              = TL_TOKENIZER_NAME or "gpt2",
        clt_path                = BEST_CHECKPOINT,
        latent_cache_path       = AUTOINTERP_DIR,
        dataset_path            = DATASET_PATH,
        disk                    = DISK,
        context_size            = CONTEXT_SIZE,
        total_autointerp_tokens = 2_000_000,
        train_batch_size_tokens = 8192,
    )

    autointerp = AutoInterp(autointerp_cfg)
    autointerp.run(AUTOINTERP_DIR)
    print(f"AutoInterp results saved to: {AUTOINTERP_DIR}")
else:
    print("AutoInterp skipped. Set RUN_AUTOINTERP = True to enable.")

---
## Cell 13 — (Optional) Compute Attribution Graph

Set `RUN_ATTRIBUTION = True` and supply a prompt relevant to your training data.

In [ ]:
RUN_ATTRIBUTION    = False
ATTRIBUTION_PROMPT = 'The opposite of "large" is '

if RUN_ATTRIBUTION:
    from clt_forge import AttributionRunner

    runner = AttributionRunner(
        model_name = TL_TOKENIZER_NAME or "gpt2",
        clt_path   = BEST_CHECKPOINT,
    )

    graph = runner.run(
        input_str = ATTRIBUTION_PROMPT,
        folder    = ATTRIBUTION_GRAPH_DIR,
    )

    print(f"Attribution graph saved to: {ATTRIBUTION_GRAPH_DIR}")
    print(f"Nodes: {len(graph.nodes)}, Edges: {len(graph.edges)}")
else:
    print("Attribution graph skipped. Set RUN_ATTRIBUTION = True to enable.")

---
## Cell 14 — (Optional) Launch Visual Interface

Opens a Dash web server at **http://127.0.0.1:8050**.
Set `LAUNCH_UI = True` to enable.

In [ ]:
LAUNCH_UI = False

if LAUNCH_UI:
    from clt_forge.frontend import main, AppConfig

    ui_cfg = AppConfig(
        graph_path      = ATTRIBUTION_GRAPH_DIR,
        autointerp_path = AUTOINTERP_DIR,
    )
    main(ui_cfg)
else:
    print("Visual interface skipped. Set LAUNCH_UI = True to enable.")

---
## Summary

| Cell | Stage | Key fix / note |
|------|-------|----------------|
| 1 | Install dependencies | Run once |
| 2 | **Paths and CLT hyperparameters** | Only cell to edit; `USE_LOCAL_DATASET` flag here |
| 3 | Convert NanoGPT `.bin` → HF Arrow | Run once; skipped if Arrow dataset already exists |
| 4 | Load checkpoint + read architecture | `model_args`, strips `_orig_mod.`/`module.`, detects vocab |
| 5 | Convert weights to TL format | All biases use `.get()` with zero fallback for `bias=False` models |
| 6 | Build HookedTransformer + verify | Missing keys = causal mask buffers, expected and OK |
| 7 | Move to device, freeze | Generation check gated on tokeniser type |
| 8 | **Build CLT config** | `distributed_setup='None'`, `adam_beta1=0.0`, `disk=DISK`, no invalid fields |
| 9 | Cache MLP activations | Skips if files exist |
| 10 | **Train the CLT** | JumpReLU + L1 sparsity |
| 11 | Evaluate | Explained variance per layer |
| 12 | AutoInterp | Optional |
| 13 | Attribution graph | Optional |
| 14 | Visual interface | Optional |

### Key parameters to tune

| Parameter | Cell | Guidance |
|-----------|------|----------|
| `EXPANSION_FACTOR` | 2 | Start at 16 for n_embd=128; increase for better features |
| `TOTAL_TRAINING_TOKENS` | 2 | More = better features. 5M is a starting point |
| `L0_COEFFICIENT` | 2 | Higher = sparser. Range 1.0–4.0 |
| `LEARNING_RATE` | 2 | 4e-4 for fp32; 8e-4 for bfloat16 |
| `CONTEXT_SIZE` | 2 | Must be <= BLOCK_SIZE. Also used as chunk size in Cell 3 |